# A6 — Preloaded Bolt Fatigue in Flange Joints (VDI 2230)

**Reference:** Okorn et al. (2021), *Analysis of Additional Load and Fatigue Life of Preloaded Bolts in a Flange Joint Considering a Bolt Bending Load*, Metals 11, 449.  
**Fatigue criterion:** Eurocode 3, detail category 50 (M20 bolts, class 8.8)  
**Verification case:** DN40 flange, M20 bolts, F_V = 50 kN, hard steel insert  

---

## Pipeline overview

```
[Geometry + Material]
        |
        v
  Node 1: Bolt compliance δ_S
        |
        v
  Node 2: Clamped parts compliance δ_P (pressure cone model)
        |
        v
  Node 3: Eccentric load factor φ*_en  (eccentricity correction)
        |
        v
  Node 4: Additional bolt force  F_SA = φ*_en · F_A
        |
        v
  Node 5: Stress range  σ_abr = σ_SAt (tension) + σ_SAb* (bending)
        |
        v
  Node 6: Fatigue life  N_R  (Eurocode 3, detail 50)
```

**Key insight:** In eccentric flange joints, bolt bending contributes 6–18× more to the stress range than axial tension. A longer bolt (HW) bends less → significantly longer fatigue life.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.size': 11,
})

## Cell 1 — Input parameters

Two configurations from Okorn 2021 (DN40 flange, M20 hard-insert case):

| Config | Bolt | Washer height | δ_S [mm/N] |
|--------|------|---------------|------------|
| HW-HI | M20×130 | High washer | 3.06e-6 |
| LW-HI | M20×90  | Low washer  | 1.53e-6 |

**Note on compliance values:** δ_S, δ_P, δ_P\*, δ_P\*\* are taken directly from Okorn (2021) supplementary data. Their detailed section-by-section derivation requires all dimensions from Figure 2 of the paper and the Excel supplement. The formulae are implemented in the functions below for general use.

**Note on β_P/β_S:** the bending compliance ratio is back-calculated from Table 1 of Okorn (2021) using the formula in Node 5. It encodes the relative bending stiffness of clamped parts vs bolt.

In [ ]:
# ── Cell 1 — Input parameters ─────────────────────────────────────────────

# Shared thread / material parameters (M20, pitch 2.5 mm, grade 8.8)
d   = 20.0        # nominal bolt diameter [mm]
P   =  2.5        # thread pitch [mm]
E_S = 210_000.0   # Young's modulus bolt   [MPa]
E_P = 210_000.0   # Young's modulus clamped parts [MPa]

# Derived thread geometry (ISO metric)
d_2 = d - 0.6495 * P          # pitch diameter [mm]
d_3 = d - 1.2269 * P          # minor diameter [mm]
d_S = (d_2 + d_3) / 2         # mean stress diameter [mm]
A_S = np.pi / 4 * d_S**2      # tensile stress area [mm²]
W_S = np.pi / 32 * d_S**3     # section modulus (threaded cross-section) [mm³]

print("=== Thread geometry (ISO M20×2.5) ===")
print(f"  d_2 = {d_2:.3f} mm   (pitch diameter)")
print(f"  d_3 = {d_3:.3f} mm   (minor diameter)")
print(f"  d_S = {d_S:.3f} mm   (mean stress diameter)")
print(f"  A_S = {A_S:.1f} mm²  (tensile stress area) — paper: 245 mm²")
print(f"  W_S = {W_S:.1f} mm³  (section modulus)     — paper: 540 mm³")

# Flange / loading geometry
a     = 31.0   # eccentricity of external load from bolt axis [mm]
s_sym =  3.5   # eccentricity of preload resultant from bolt axis [mm]
F_V   = 50_000 # bolt preload [N]

# ── HW-HI configuration (High Washer, hard insert) ────────────────────────
HW = {
    'name'    : 'HW-HI (M20×130)',
    # Compliances from Okorn 2021 supplementary [mm/N]
    'delta_S' : 3.06e-6,   # bolt compliance
    'delta_P' : 1.07e-6,   # clamped parts compliance (axial)
    'delta_Ps': 1.07e-6,   # δ_P* — corrected for preload eccentricity
    'delta_Pss':1.07e-6,   # δ_P** — corrected for load eccentricity
    # Load introduction factor (VDI 2230 table, inner load introduction)
    'n'       : 0.10,
    # Bending compliance ratio β_P/β_S (back-calculated from Table 1, Okorn 2021)
    # β_P/β_S ≈ 0.01213
    'beta_ratio': 0.01213,
}

# ── LW-HI configuration (Low Washer, hard insert) ─────────────────────────
LW = {
    'name'    : 'LW-HI (M20×90)',
    'delta_S' : 1.53e-6,
    'delta_P' : 3.19e-7,
    'delta_Ps': 1.29e-7,
    'delta_Pss':1.31e-7,
    'n'       : 0.30,
    'beta_ratio': 0.02678,
}

configs = [HW, LW]

print("\n=== Configuration inputs ===")
for c in configs:
    print(f"\n  [{c['name']}]")
    print(f"    δ_S  = {c['delta_S']:.2e} mm/N")
    print(f"    δ_P  = {c['delta_P']:.2e} mm/N")
    print(f"    δ_P* = {c['delta_Ps']:.2e} mm/N")
    print(f"    δ_P**= {c['delta_Pss']:.2e} mm/N")
    print(f"    n    = {c['n']:.2f}")
    print(f"    β_P/β_S = {c['beta_ratio']:.5f}")

## Cell 2 — Nodes 1 & 2: Compliance formulae

**Node 1 — Bolt compliance (springs in series):**

$$\delta_S = \frac{1}{E_S}\left(\frac{0.4d}{A_N} + \frac{l_1}{A_1} + \frac{l_2}{A_2} + \frac{l_{\mathrm{Gew}}}{A_3} + \frac{0.5d}{A_3} + \frac{0.4d}{A_N}\right)$$

The `0.4d` and `0.5d` terms are VDI empirical additions for head/nut and thread run-out zones.

**Node 2 — Clamped parts compliance (pressure cone, VDI 2230 §5):**

Full cone: $D_A \geq d_w + w \, l_K \tan\varphi$
$$\delta_F = \frac{2\ln\!\left[\dfrac{(d_w+d_h)(d_w + w l_K \tan\varphi - d_h)}{(d_w-d_h)(d_w + w l_K \tan\varphi + d_h)}\right]}{w \, E_P \, \pi \, d_h \tan\varphi}$$

Truncated cone: $d_w < D_A < d_w + w \, l_K \tan\varphi$
$$\delta_F = \frac{1}{E_P \pi}\left[\frac{2}{d_h \tan\varphi}\ln\frac{(d_w+d_h)(D_A-d_h)}{(d_w-d_h)(D_A+d_h)} + \frac{4}{D_A^2 - d_h^2}\left(l_K - \frac{D_A - d_w}{w \tan\varphi}\right)\right]$$

In [ ]:
# ── Cell 2 — Compliance formulae (Nodes 1 & 2) ───────────────────────────

def bolt_compliance(E_S, d, A_N, l_sections, A_sections,
                    l_Gew, A_3):
    """
    Bolt compliance δ_S — springs in series (VDI 2230 Eq. 3).

    Parameters
    ----------
    E_S        : Young's modulus bolt [MPa]
    d          : nominal diameter [mm]
    A_N        : tensile stress area [mm²]
    l_sections : list of shank section lengths [mm]
    A_sections : list of shank section areas  [mm²]
    l_Gew      : free thread length [mm]
    A_3        : minor area [mm²]

    Returns
    -------
    delta_S : bolt compliance [mm/N]
    """
    head_nut  = 0.4 * d / A_N   # head contribution (Eq. 3 Okorn 2021)
    shank     = sum(l / A for l, A in zip(l_sections, A_sections))
    thread    = l_Gew / A_3
    runout    = 0.5 * d / A_3   # thread run-out contribution
    tail      = 0.4 * d / A_N   # nut contribution
    return (head_nut + shank + thread + runout + tail) / E_S


def clamped_compliance_full_cone(E_P, d_w, d_h, l_K, phi_deg=30.0, w=1):
    """
    Flange compliance — fully developed pressure cone (VDI 2230 Eq. 13).

    Parameters
    ----------
    E_P     : Young's modulus clamped parts [MPa]
    d_w     : bearing surface diameter (wrench flat) [mm]
    d_h     : hole diameter [mm]
    l_K     : clamping length [mm]
    phi_deg : cone half-angle [deg], default 30°
    w       : 1 = nut, 2 = tapped hole

    Returns
    -------
    delta_F : flange compliance [mm/N]
    """
    tan_phi = np.tan(np.radians(phi_deg))
    spread  = d_w + w * l_K * tan_phi
    num = (d_w + d_h) * (spread - d_h)
    den = (d_w - d_h) * (spread + d_h)
    return (2 * np.log(num / den)) / (w * E_P * np.pi * d_h * tan_phi)


def clamped_compliance_truncated_cone(E_P, d_w, d_h, D_A, l_K,
                                       phi_deg=30.0, w=1):
    """
    Flange compliance — truncated pressure cone (VDI 2230 Eq. 14).
    Use when d_w < D_A < d_w + w*l_K*tan(phi).
    """
    tan_phi = np.tan(np.radians(phi_deg))
    term1 = (2 / (d_h * tan_phi)) * np.log(
        (d_w + d_h) * (D_A - d_h) / ((d_w - d_h) * (D_A + d_h))
    )
    term2 = (4 / (D_A**2 - d_h**2)) * (l_K - (D_A - d_w) / (w * tan_phi))
    return (term1 + term2) / (E_P * np.pi)


def washer_compliance(E_W, l_W, D_Aw, d_h):
    """
    Washer / insert compliance δ_W (VDI 2230 Eq. 15).
    δ_W = 4*l_W / (E_W * π * (D_Aw² - d_h²))
    """
    return 4 * l_W / (E_W * np.pi * (D_Aw**2 - d_h**2))


print("Compliance functions defined (Nodes 1 & 2).")
print("For this verification case, δ_S, δ_P, δ_P*, δ_P** are taken")
print("directly from Okorn (2021) supplementary data (see Cell 1).")

## Cell 3 — Node 3: Eccentric load factor φ\*_en

$$\varphi^*_{en} = n \cdot \frac{\delta_P^{**}}{\delta_S + \delta_P^*}$$

- $n$ = load introduction factor (from VDI 2230 tables; depends on where the load enters the joint)
- $\delta_P^* = \delta_P + s_{\mathrm{sym}}^2 \sum \dfrac{l_i}{E_{P,i}\, I_{\mathrm{Bers},i}}$ — compliance corrected for preload eccentricity $s_{\mathrm{sym}}$
- $\delta_P^{**} = \delta_P + a\, s_{\mathrm{sym}} \sum \dfrac{l_i}{E_{P,i}\, I_{\mathrm{Bers},i}}$ — compliance corrected for load eccentricity $a$

The eccentricity terms add a bending-to-axial coupling: when the external load is offset from the bolt axis, the joint rotates and the bolt sees an amplified displacement.

In [ ]:
# ── Cell 3 — Node 3: Eccentric load factor φ*_en ─────────────────────────
# Eq. (10) Okorn 2021

def phi_eccentric(n, delta_Pss, delta_S, delta_Ps):
    """Eccentric load factor φ*_en."""
    return n * delta_Pss / (delta_S + delta_Ps)


print("=" * 60)
print(f"{'Parameter':<20}  {'HW-HI':>10}  {'LW-HI':>10}  Paper")
print("=" * 60)

for c in configs:
    c['phi_en'] = phi_eccentric(
        c['n'], c['delta_Pss'], c['delta_S'], c['delta_Ps']
    )

print(f"{'n (load factor)':<20}  {HW['n']:>10.2f}  {LW['n']:>10.2f}")
print(f"{'δ_S [mm/N]':<20}  {HW['delta_S']:>10.2e}  {LW['delta_S']:>10.2e}")
print(f"{'δ_P* [mm/N]':<20}  {HW['delta_Ps']:>10.2e}  {LW['delta_Ps']:>10.2e}")
print(f"{'δ_P** [mm/N]':<20}  {HW['delta_Pss']:>10.2e}  {LW['delta_Pss']:>10.2e}")
print("-" * 60)
print(f"{'φ*_en (calculated)':<20}  {HW['phi_en']:>10.4f}  {LW['phi_en']:>10.4f}")
print(f"{'φ*_en (paper)':<20}  {'0.026':>10}  {'0.021':>10}")

print("\n--- Comparison: centric vs eccentric load factor ---")
for c in configs:
    phi_centric = c['n'] * c['delta_P'] / (c['delta_S'] + c['delta_P'])
    print(f"  [{c['name']}]  φ_en (centric) = {phi_centric:.4f}  "
          f"φ*_en (eccentric) = {c['phi_en']:.4f}")

## Cell 4 — Nodes 4 & 5: Additional force and stress range

**Node 4 — Additional bolt force:**
$$F_{SA} = \varphi^*_{en} \cdot F_A$$

**Node 5 — Stress range (tension + bending):**
$$\sigma_{SA,t} = \frac{F_{SA}}{A_S} \qquad \text{(tension component)}$$

$$\sigma^*_{SA,b} = \frac{\beta_P}{\beta_S}\left(1 - \frac{s_{\mathrm{sym}}}{a}\,\varphi^*_{en}\right)\frac{F_A \cdot a}{W_S} \qquad \text{(bending component)}$$

$$\sigma_{abr} = \sigma_{SA,t} + \sigma^*_{SA,b}$$

The ratio $\beta_P / \beta_S$ (bending compliance ratio) controls how much of the external moment is transferred to the bolt. A stiffer flange (high $\beta_P/\beta_S$) means more bolt bending.

Note that the bending term is typically **6–18× larger** than the tension term for this flange geometry.

In [ ]:
# ── Cell 4 — Nodes 4 & 5: F_SA and stress range ──────────────────────────
# Eq. (11), (16), (17) Okorn 2021

F_A_ref = 13_600  # N — reference load for Table 1 verification

def additional_force(phi_en, F_A):
    """Node 4: Additional bolt force F_SA [N]. Eq. (11) Okorn 2021."""
    return phi_en * F_A

def stress_range(phi_en, F_A, delta_Pss, delta_S, delta_Ps,
                 beta_ratio, a, s_sym, A_S, W_S):
    """
    Node 5: Total stress range σ_abr [MPa]. Eq. (16)+(17) Okorn 2021.

    Returns
    -------
    sigma_t  : tension component [MPa]
    sigma_b  : bending component [MPa]
    sigma_abr: total stress range [MPa]
    """
    F_SA    = additional_force(phi_en, F_A)          # Eq. (11)
    sigma_t = F_SA / A_S                             # Eq. (16) — tension
    sigma_b = (beta_ratio
               * (1 - s_sym / a * phi_en)
               * F_A * a / W_S)                      # Eq. (17) — bending
    return sigma_t, sigma_b, sigma_t + sigma_b


# ── Print verification table ───────────────────────────────────────────────
print(f"Verification at F_A = {F_A_ref/1000:.2f} kN")
print("=" * 68)
print(f"{'Parameter':<22} {'HW-HI calc':>10} {'HW paper':>10} {'LW-HI calc':>10} {'LW paper':>10}")
print("=" * 68)

for c in configs:
    sigma_t, sigma_b, sigma_abr = stress_range(
        c['phi_en'], F_A_ref,
        c['delta_Pss'], c['delta_S'], c['delta_Ps'],
        c['beta_ratio'], a, s_sym, A_S, W_S
    )
    c['sigma_t_ref']  = sigma_t
    c['sigma_b_ref']  = sigma_b
    c['sigma_abr_ref']= sigma_abr
    F_SA = additional_force(c['phi_en'], F_A_ref)
    c['F_SA_ref']     = F_SA

print(f"{'φ*_en':<22} {HW['phi_en']:>10.4f} {'0.026':>10} {LW['phi_en']:>10.4f} {'0.021':>10}")
print(f"{'F_SA [kN]':<22} {HW['F_SA_ref']/1000:>10.3f} {'0.356':>10} {LW['F_SA_ref']/1000:>10.3f} {'0.281':>10}")
print(f"{'σ_SAt [MPa]':<22} {HW['sigma_t_ref']:>10.2f} {'1.45':>10} {LW['sigma_t_ref']:>10.2f} {'1.15':>10}")
print(f"{'σ*_SAb [MPa]':<22} {HW['sigma_b_ref']:>10.2f} {'9.44':>10} {LW['sigma_b_ref']:>10.2f} {'20.86':>10}")
print(f"{'σ_abr [MPa]':<22} {HW['sigma_abr_ref']:>10.2f} {'10.89':>10} {LW['sigma_abr_ref']:>10.2f} {'22.01':>10}")
print("-" * 68)
print(f"{'Bending / tension ratio':<22} {HW['sigma_b_ref']/HW['sigma_t_ref']:>10.1f}x {'':>10} {LW['sigma_b_ref']/LW['sigma_t_ref']:>10.1f}x")
print()
print("→ Bending dominates by 6–18×. Tension alone severely underestimates fatigue damage.")

## Cell 5 — Node 6: Fatigue life (Eurocode 3, detail 50)

For M20 class 8.8 bolts, Eurocode 3 assigns **detail category 50** (ΔσC = 50 MPa at N = 2×10⁶ cycles).

The S-N curve has two slopes:

$$
N_R = \begin{cases}
\dfrac{C_3}{\sigma_{abr}^3} & \sigma_{abr} \geq \sigma_{D} = 36.84\ \mathrm{MPa} \\[8pt]
\dfrac{C_5}{\sigma_{abr}^5} & \sigma_{L} \leq \sigma_{abr} < \sigma_{D} \\[8pt]
\infty & \sigma_{abr} < \sigma_{L} = 20.23\ \mathrm{MPa}
\end{cases}
$$

Transition points (derived from EC3 clause 9):

$$\sigma_D = 50 \left(\frac{2 \times 10^6}{5 \times 10^6}\right)^{1/3} = 36.84\ \mathrm{MPa}$$

$$\sigma_L = 36.84 \left(\frac{5 \times 10^6}{10^8}\right)^{1/5} = 20.23\ \mathrm{MPa}$$

$$C_3 = 50^3 \times 2 \times 10^6 = 2.5 \times 10^{11} \qquad C_5 = 36.84^5 \times 5 \times 10^6 = 3.392 \times 10^{14}$$

In [ ]:
# ── Cell 5 — Node 6: Fatigue life ────────────────────────────────────────
# Eurocode 3 detail 50 — Eq. (18)+(19) Okorn 2021

# ── Transition points (derived from EC3 clause 9) ─────────────────────────
delta_C   = 50.0       # detail category [MPa]
N_C       = 2e6        # reference cycles
N_D       = 5e6        # knee point cycles
N_L       = 1e8        # cut-off cycles

sigma_D   = delta_C * (N_C / N_D) ** (1/3)      # 36.84 MPa
sigma_L   = sigma_D * (N_D / N_L) ** (1/5)      # 20.23 MPa
C3        = delta_C**3 * N_C                     # 2.5e11
C5        = sigma_D**5 * N_D                     # 3.392e14

print("=== EC3 detail 50 — S-N parameters ===")
print(f"  σ_D (m=3 → m=5 knee)   = {sigma_D:.2f} MPa")
print(f"  σ_L (cut-off)          = {sigma_L:.2f} MPa")
print(f"  C3 = Δσ_C³·N_C         = {C3:.3e}")
print(f"  C5 = σ_D⁵·N_D          = {C5:.3e}")


def fatigue_life(sigma_abr, sigma_D=sigma_D, sigma_L=sigma_L, C3=C3, C5=C5):
    """Fatigue life N_R [cycles] for EC3 detail 50. Eq. (18) Okorn 2021."""
    if sigma_abr >= sigma_D:
        return C3 / sigma_abr**3
    elif sigma_abr >= sigma_L:
        return C5 / sigma_abr**5
    else:
        return np.inf

# Vectorised version for plotting
fatigue_life_vec = np.vectorize(fatigue_life)


# ── Eight load levels from Tables 4–5 of Okorn 2021 ──────────────────────
F_A_levels_kN = np.array([6.84, 13.68, 20.53, 27.37, 34.20, 41.05, 47.89, 54.74])
F_A_levels_N  = F_A_levels_kN * 1000

# Expected values from paper (Tables 4–5)
HW_paper = {
    'sigma': [5.44, 10.89, 16.33, 21.77, 27.22, 32.66, 38.10, 43.55],
    'N_R'  : [np.inf, np.inf, np.inf, 6.93e7, 2.27e7, 9.13e6, 4.52e6, 3.03e6]
}
LW_paper = {
    'sigma': [11.00, 22.01, 32.99, 43.98, 54.98, 65.97, 76.97, 87.96],
    'N_R'  : [np.inf, 6.60e7, 8.69e6, 2.94e6, 1.50e6, 8.71e5, 5.48e5, 3.67e5]
}


def compute_series(config, F_A_arr):
    """Compute σ_abr and N_R for an array of applied loads."""
    sigmas = []
    N_Rs   = []
    for F_A in F_A_arr:
        _, _, sigma_abr = stress_range(
            config['phi_en'], F_A,
            config['delta_Pss'], config['delta_S'], config['delta_Ps'],
            config['beta_ratio'], a, s_sym, A_S, W_S
        )
        sigmas.append(sigma_abr)
        N_Rs.append(fatigue_life(sigma_abr))
    return np.array(sigmas), np.array(N_Rs)


HW['sigmas'], HW['N_Rs'] = compute_series(HW, F_A_levels_N)
LW['sigmas'], LW['N_Rs'] = compute_series(LW, F_A_levels_N)


# ── Print comparison tables ────────────────────────────────────────────────
def fmt_NR(N):
    return 'INF' if np.isinf(N) else f'{N:.2e}'

print("\n=== HW-HI: σ_abr and N_R vs applied load ===")
print(f"  {'F_A [kN]':>8} | {'σ_abr calc':>10} | {'σ_abr paper':>11} | {'N_R calc':>10} | {'N_R paper':>10}")
print("  " + "-"*62)
for i, (F, s_c, N_c, s_p, N_p) in enumerate(zip(
        F_A_levels_kN, HW['sigmas'], HW['N_Rs'],
        HW_paper['sigma'], HW_paper['N_R'])):
    print(f"  {F:>8.2f} | {s_c:>10.2f} | {s_p:>11.2f} | {fmt_NR(N_c):>10} | {fmt_NR(N_p):>10}")

print("\n=== LW-HI: σ_abr and N_R vs applied load ===")
print(f"  {'F_A [kN]':>8} | {'σ_abr calc':>10} | {'σ_abr paper':>11} | {'N_R calc':>10} | {'N_R paper':>10}")
print("  " + "-"*62)
for i, (F, s_c, N_c, s_p, N_p) in enumerate(zip(
        F_A_levels_kN, LW['sigmas'], LW['N_Rs'],
        LW_paper['sigma'], LW_paper['N_R'])):
    print(f"  {F:>8.2f} | {s_c:>10.2f} | {s_p:>11.2f} | {fmt_NR(N_c):>10} | {fmt_NR(N_p):>10}")

## Cell 6 — Plot 1: Stress range vs applied load

In [ ]:
# ── Cell 6 — Plot 1: σ_abr vs F_A ────────────────────────────────────────

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(F_A_levels_kN, HW['sigmas'], 'o-', color='steelblue',
        label='HW-HI (M20×130, long bolt)', linewidth=2, markersize=6)
ax.plot(F_A_levels_kN, LW['sigmas'], 's--', color='tomato',
        label='LW-HI (M20×90, short bolt)', linewidth=2, markersize=6)

# S-N region boundaries
x_lim = ax.get_xlim()
ax.axhline(sigma_D, color='darkorange', linestyle=':', linewidth=1.5,
           label=f'σ_D = {sigma_D:.1f} MPa  (m=3 / m=5 knee)')
ax.axhline(sigma_L, color='seagreen',   linestyle=':', linewidth=1.5,
           label=f'σ_L = {sigma_L:.1f} MPa  (cut-off, infinite life)')

# Region shading
y_max = max(LW['sigmas'].max(), sigma_D) * 1.15
ax.fill_betweenx([sigma_D, y_max],  0, 60, alpha=0.04, color='tomato')
ax.fill_betweenx([sigma_L, sigma_D],0, 60, alpha=0.06, color='orange')
ax.fill_betweenx([0, sigma_L],      0, 60, alpha=0.06, color='seagreen')

ax.text(55, (sigma_D + y_max)/2,        'm = 3 region',    ha='right', color='tomato',     fontsize=9)
ax.text(55, (sigma_L + sigma_D)/2,      'm = 5 region',    ha='right', color='darkorange',  fontsize=9)
ax.text(55, sigma_L / 2,                'Infinite life',   ha='right', color='seagreen',    fontsize=9)

ax.set_xlabel('Applied load F_A [kN]')
ax.set_ylabel('Stress range σ_abr [MPa]')
ax.set_title('A6 — Stress range vs applied load\nOkorn et al. (2021), Metals 11, 449')
ax.set_xlim(0, 60)
ax.set_ylim(0, y_max)
ax.legend(loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig('a6_stress_range_vs_load.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: a6_stress_range_vs_load.png")

## Cell 7 — Plot 2: Fatigue life vs applied load (semi-log)

In [ ]:
# ── Cell 7 — Plot 2: N_R vs F_A (semi-log) ───────────────────────────────

# Replace inf with a plot ceiling for visualisation
N_plot_ceil = 1e9
HW_NR_plot  = np.where(np.isinf(HW['N_Rs']), N_plot_ceil, HW['N_Rs'])
LW_NR_plot  = np.where(np.isinf(LW['N_Rs']), N_plot_ceil, LW['N_Rs'])

fig, ax = plt.subplots(figsize=(8, 5))

ax.semilogy(F_A_levels_kN, HW_NR_plot, 'o-', color='steelblue',
            label='HW-HI (long bolt)', linewidth=2, markersize=6)
ax.semilogy(F_A_levels_kN, LW_NR_plot, 's--', color='tomato',
            label='LW-HI (short bolt)', linewidth=2, markersize=6)

# Mark finite-life threshold
ax.axhline(N_D, color='grey', linestyle=':', linewidth=1)
ax.text(55, N_D * 1.3, f'{N_D:.0e} cycles', ha='right', color='grey', fontsize=9)

# Arrow annotations for 'infinite life' points
inf_hw = F_A_levels_kN[np.isinf(HW['N_Rs'])]
inf_lw = F_A_levels_kN[np.isinf(LW['N_Rs'])]
for fx in inf_hw:
    ax.annotate('∞', (fx, N_plot_ceil), xytext=(0, 8), textcoords='offset points',
                ha='center', color='steelblue', fontsize=11)
for fx in inf_lw:
    ax.annotate('∞', (fx, N_plot_ceil), xytext=(0, 8), textcoords='offset points',
                ha='center', color='tomato', fontsize=11)

ax.set_xlabel('Applied load F_A [kN]')
ax.set_ylabel('Fatigue life N_R [cycles]')
ax.set_title('A6 — Fatigue life vs applied load\n'
             'High washer (long bolt) dramatically extends fatigue life')
ax.set_xlim(0, 60)
ax.set_ylim(1e5, 2e9)
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(ticker.LogFormatterMathtext())

plt.tight_layout()
plt.savefig('a6_fatigue_life_vs_load.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: a6_fatigue_life_vs_load.png")

## Cell 8 — Plot 3: Wöhler curve (EC3 detail 50) with operating points

In [ ]:
# ── Cell 8 — Plot 3: Wöhler / S-N curve with operating points ────────────

# Build the S-N curve
N_curve    = np.logspace(5, 9, 400)
sigma_curve = np.zeros_like(N_curve)

for i, N in enumerate(N_curve):
    if N <= N_D:
        sigma_curve[i] = (C3 / N) ** (1/3)
    elif N <= N_L:
        sigma_curve[i] = (C5 / N) ** (1/5)
    else:
        sigma_curve[i] = sigma_L

fig, ax = plt.subplots(figsize=(8, 5))

# S-N line
ax.loglog(N_curve, sigma_curve, 'k-', linewidth=2, label='EC3 detail 50 (M20 class 8.8)')

# Knee points
ax.scatter([N_D, N_L], [sigma_D, sigma_L], color='k', zorder=5, s=40)
ax.annotate(f'  σ_D = {sigma_D:.1f} MPa\n  N = {N_D:.0e}',
            (N_D, sigma_D), fontsize=8, va='bottom')
ax.annotate(f'  σ_L = {sigma_L:.1f} MPa\n  N = {N_L:.0e}',
            (N_L, sigma_L), fontsize=8, va='top')

# Operating points
# Filter out infinite life (no point on log-log)
hw_finite = ~np.isinf(HW['N_Rs'])
lw_finite = ~np.isinf(LW['N_Rs'])

ax.scatter(HW['N_Rs'][hw_finite], HW['sigmas'][hw_finite],
           color='steelblue', marker='o', s=60, zorder=6,
           label='HW-HI operating points')
ax.scatter(LW['N_Rs'][lw_finite], LW['sigmas'][lw_finite],
           color='tomato',    marker='s', s=60, zorder=6,
           label='LW-HI operating points')

# Infinite life zone
ax.axhline(sigma_L, color='seagreen', linestyle='--', linewidth=1, alpha=0.6)
ax.fill_between([1e5, 1e9], [0, 0], [sigma_L, sigma_L],
                alpha=0.05, color='seagreen')
ax.text(1e8, sigma_L * 0.7, 'Infinite life region',
        ha='center', color='seagreen', fontsize=9)

ax.set_xlabel('Fatigue life N_R [cycles]')
ax.set_ylabel('Stress range σ_abr [MPa]')
ax.set_title('A6 — Wöhler curve (EC3 detail 50) with operating points\n'
             'HW-HI points cluster near infinite life; LW-HI penetrate the finite-life region')
ax.set_xlim(1e5, 1e9)
ax.set_ylim(5, 150)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('a6_woehler_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: a6_woehler_curve.png")

## Cell 9 — Parametric analysis: effect of bolt compliance on stress range

This analysis varies $\delta_S$ while holding $\delta_P^*$, $\delta_P^{**}$, $n$ constant.

**Scope and limitation:** This parametric sweep captures only the **tension component** effect cleanly:

$$\sigma_{SA,t} = \frac{n \cdot \delta_P^{**}}{\delta_S + \delta_P^*} \cdot \frac{F_A}{A_S}$$

The **bending component** depends on $\beta_P/\beta_S$, which is itself a function of the bolt length and cross-section distribution — so it changes non-trivially as $\delta_S$ varies. In the sweep below, $\beta_P/\beta_S$ is held fixed at its reference value for each configuration. The result correctly shows the trend but underestimates the change in $\sigma_{abr}$ for large deviations from the reference geometry.

In [ ]:
# ── Cell 9 — Parametric: δ_S sweep ───────────────────────────────────────

F_A_param    = 27_370  # N — representative finite-life load
delta_S_arr  = np.linspace(0.5e-6, 6.0e-6, 300)  # mm/N

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)

for ax, c, color, label_tag in zip(axes, [HW, LW],
                                   ['steelblue', 'tomato'],
                                   ['HW-HI reference', 'LW-HI reference']):
    sigma_t_arr  = []
    sigma_b_arr  = []
    sigma_tot_arr= []

    for dS in delta_S_arr:
        # Update phi_en for each δ_S value
        phi_en_i = phi_eccentric(c['n'], c['delta_Pss'], dS, c['delta_Ps'])
        st, sb, sa = stress_range(
            phi_en_i, F_A_param,
            c['delta_Pss'], dS, c['delta_Ps'],
            c['beta_ratio'], a, s_sym, A_S, W_S
        )
        sigma_t_arr.append(st)
        sigma_b_arr.append(sb)
        sigma_tot_arr.append(sa)

    sigma_t_arr   = np.array(sigma_t_arr)
    sigma_b_arr   = np.array(sigma_b_arr)
    sigma_tot_arr = np.array(sigma_tot_arr)

    ax.plot(delta_S_arr * 1e6, sigma_tot_arr, color=color, linewidth=2, label='σ_abr (total)')
    ax.plot(delta_S_arr * 1e6, sigma_t_arr,   color=color, linewidth=1.2,
            linestyle='--', label='σ_SAt (tension only)')
    ax.plot(delta_S_arr * 1e6, sigma_b_arr,   color=color, linewidth=1.2,
            linestyle=':', label='σ*_SAb (bending only)')

    # Mark reference δ_S
    ax.axvline(c['delta_S'] * 1e6, color='grey', linestyle=':', linewidth=1)
    ax.annotate(label_tag, (c['delta_S'] * 1e6, sigma_tot_arr.max() * 0.95),
                xytext=(5, 0), textcoords='offset points', fontsize=8, color='grey')

    # EC3 thresholds
    ax.axhline(sigma_D, color='darkorange', linestyle=':', linewidth=1, alpha=0.7)
    ax.axhline(sigma_L, color='seagreen',   linestyle=':', linewidth=1, alpha=0.7)

    ax.set_xlabel('Bolt compliance δ_S [×10⁻⁶ mm/N]')
    ax.set_ylabel('Stress range [MPa]')
    ax.set_title(f"{c['name']}\nF_A = {F_A_param/1000:.2f} kN")
    ax.legend(fontsize=8)

fig.suptitle('A6 — Parametric: bolt compliance δ_S vs stress components\n'
             '(β_P/β_S held fixed — bending trend approximate; see note above)',
             fontsize=11)
plt.tight_layout()
plt.savefig('a6_parametric_compliance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: a6_parametric_compliance.png")

print("\n=== Physical interpretation ===")
print("A longer bolt has higher δ_S. Higher δ_S → lower φ*_en → lower F_SA → lower σ_SAt.")
print("The bending component (held fixed here) also decreases in reality because")
print("a longer bolt has a higher β_S, reducing β_P/β_S.")
print("Both effects compound: high washer (longer bolt) is doubly beneficial.")

---

## Summary

| Result | HW-HI | LW-HI | Ratio |
|--------|--------|--------|-------|
| σ_abr at F_A = 27.37 kN [MPa] | 21.77 | 43.98 | 2.0× |
| N_R at F_A = 27.37 kN [cycles] | 6.93×10⁷ | 2.94×10⁶ | **24×** |
| Bending / tension ratio | 6.5× | 18.1× | — |

**Key takeaways:**

1. **Bending dominates.** The bending stress component is 6–18× larger than the tension component. Ignoring eccentricity leads to a severe non-conservative error.
2. **Bolt length matters.** The high washer (longer bolt, higher δ_S) reduces both φ\*_en and β_P/β_S simultaneously, cutting the stress range by ~2× and multiplying fatigue life by ~24×.
3. **The cut-off matters.** The HW-HI configuration stays below σ_L at low loads → theoretically infinite life. LW-HI enters the finite-life zone at F_A ≈ 13.7 kN.

---

**Reference:** Okorn, I.; Nagode, M.; Klemenc, J.; Oman, S. *Analysis of Additional Load and Fatigue Life of Preloaded Bolts in a Flange Joint Considering a Bolt Bending Load.* Metals 2021, 11, 449. https://doi.org/10.3390/met11030449

**Fatigue criterion:** EN 1993-1-9 (Eurocode 3), detail category 50, M20 bolts class 8.8.

**Tool:** eng-tools.dev · Serie A · A6